# CSE440 Final Project (Redone using Lab Materials)

This notebook is rebuilt using the approach style from the course labs:
- Lab1: NLTK + EDA workflow
- Lab2: word representations (TF-IDF, Skip-gram)
- Lab3: Pandas + Keras pipeline style
- Lab4: RNN family model training patterns

All required experiments are included for three preprocessing settings.

## 1) Imports and Setup

In [ ]:
# If needed, uncomment to install dependencies
# !pip install -q pandas numpy scikit-learn matplotlib seaborn nltk gensim wordcloud tensorflow

import re
import html
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from gensim.models import Word2Vec

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, SimpleRNN, GRU, LSTM, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as exc:
        print('nltk.download skipped or failed:', pkg, exc)

# After normalize_text(), tokens are letters-only words; split() avoids Punkt LookupError on some setups.

## 2) Load Dataset

In [ ]:
TRAIN_PATH = 'Training_data_9.csv'
TEST_PATH = 'Test_data.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df = train_df.rename(columns={'News Headline': 'text', 'News Topic': 'label'})
test_df = test_df.rename(columns={'News Headline': 'text', 'News Topic': 'label'})

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('\nLabel distribution in train:')
print(train_df['label'].value_counts())

train_df.head()

## 3) EDA (Lab1 style)

In [ ]:
def strip_html_noise(text):
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

eda_df = train_df.copy()
eda_df['clean_eda'] = eda_df['text'].apply(strip_html_noise)
eda_df['char_len'] = eda_df['clean_eda'].str.len()
eda_df['word_len'] = eda_df['clean_eda'].str.split().str.len()

display(eda_df[['clean_eda', 'label', 'char_len', 'word_len']].head(3))

plt.figure(figsize=(8, 4))
sns.countplot(data=eda_df, x='label', order=eda_df['label'].value_counts().index)
plt.title('Class Distribution')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(eda_df['word_len'], bins=50, kde=True, ax=ax[0])
ax[0].set_title('Word Count Distribution')
sns.boxplot(data=eda_df, x='label', y='word_len', ax=ax[1])
ax[1].set_title('Word Count by Class')
ax[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

all_text = ' '.join(eda_df['clean_eda']).lower()
wc = WordCloud(width=900, height=400, background_color='white').generate(all_text)
plt.figure(figsize=(14, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('WordCloud of Training Headlines')
plt.show()

top_tokens = Counter(all_text.split()).most_common(30)
pd.DataFrame(top_tokens, columns=['token', 'count'])

## 4) Build Three Preprocessing Variants

- **No preprocessing**: raw headlines
- **Extreme preprocessing**: HTML removal, lowercase, punctuation removal, stopwords removal, lemmatization, stemming
- **Optimum preprocessing**: moderate cleaning based on EDA, keeps negation terms for sentiment/context preservation

In [ ]:
STOPWORDS = set(stopwords.words('english'))
NEGATION_KEEP = {'no', 'not', 'nor'}
STOPWORDS_OPTIMUM = STOPWORDS - NEGATION_KEEP
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_none(text):
    return str(text)

def normalize_text(text):
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def preprocess_extreme(text):
    text = normalize_text(text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

def preprocess_optimum(text):
    text = normalize_text(text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS_OPTIMUM and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

PREPROCESSORS = {
    'none': preprocess_none,
    'extreme': preprocess_extreme,
    'optimum': preprocess_optimum
}

## 5) Split Data + Encode Labels

In [ ]:
# Set FAST_MODE=True for debugging only, False for final report runs
FAST_MODE = False
FAST_SAMPLES_PER_CLASS = 2500

work_train = train_df.copy()
work_test = test_df.copy()

if FAST_MODE:
    work_train = work_train.groupby('label', group_keys=False).apply(lambda x: x.sample(min(len(x), FAST_SAMPLES_PER_CLASS), random_state=SEED)).reset_index(drop=True)
    work_test = work_test.groupby('label', group_keys=False).apply(lambda x: x.sample(min(len(x), max(700, FAST_SAMPLES_PER_CLASS // 2)), random_state=SEED)).reset_index(drop=True)

train_part, val_part = train_test_split(
    work_train,
    test_size=0.15,
    random_state=SEED,
    stratify=work_train['label']
)

label_encoder = LabelEncoder()
label_encoder.fit(work_train['label'])

y_train = label_encoder.transform(train_part['label'])
y_val = label_encoder.transform(val_part['label'])
y_test = label_encoder.transform(work_test['label'])

num_classes = len(label_encoder.classes_)

y_train_oh = to_categorical(y_train, num_classes=num_classes)
y_val_oh = to_categorical(y_val, num_classes=num_classes)
y_test_oh = to_categorical(y_test, num_classes=num_classes)

print('Train split:', train_part.shape)
print('Val split:', val_part.shape)
print('Test split:', work_test.shape)
print('Classes:', list(label_encoder.classes_))

## 6) Shared Utility Functions

In [ ]:
all_results = []
tuning_logs = []

def get_preprocessed_splits(prep_fn):
    X_train_text = train_part['text'].apply(prep_fn).astype(str).values
    X_val_text = val_part['text'].apply(prep_fn).astype(str).values
    X_test_text = work_test['text'].apply(prep_fn).astype(str).values
    return X_train_text, X_val_text, X_test_text

def evaluate_and_store(y_true, y_pred, prep_name, rep_name, model_name):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    cm = confusion_matrix(y_true, y_pred)

    print(f'\n[{prep_name} | {rep_name} | {model_name}]')
    print(f'Accuracy: {acc:.4f}')
    print(f'Macro F1: {macro_f1:.4f}')
    print('Classification report:')
    print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix - {prep_name} | {rep_name} | {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.xticks(rotation=20)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    all_results.append({
        'preprocessing': prep_name,
        'representation': rep_name,
        'model': model_name,
        'accuracy': acc,
        'macro_f1': macro_f1
    })

def add_tuning_log(prep, rep, model, config, val_f1):
    tuning_logs.append({
        'preprocessing': prep,
        'representation': rep,
        'model': model,
        'config': str(config),
        'val_macro_f1': float(val_f1)
    })

## 7) TF-IDF + Logistic Regression (ML) + Deep NN (Keras)

In [ ]:
def build_tfidf_dnn(input_dim, num_classes, dense1=256, dense2=128, drop=0.5, lr=1e-3):
    model = Sequential([
        Dense(dense1, activation='relu', input_shape=(input_dim,)),
        Dropout(drop),
        Dense(dense2, activation='relu'),
        Dropout(drop),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

tfidf_grid = {
    'max_features': [5000, 8000],
    'ngram_range': [(1, 1), (1, 2)],
    'min_df': [2]
}

lr_grid = {
    'C': [0.5, 1.0, 2.0],
    'solver': ['lbfgs'],
    'max_iter': [400]
}

dnn_grid = {
    'dense1': [256, 384],
    'dense2': [128, 192],
    'drop': [0.35, 0.5],
    'lr': [1e-3],
    'batch_size': [64],
    'epochs': [8]
}

for prep_name, prep_fn in PREPROCESSORS.items():
    print('\n' + '=' * 95)
    print('TF-IDF experiments for preprocessing:', prep_name)

    X_train_text, X_val_text, X_test_text = get_preprocessed_splits(prep_fn)

    best_lr_f1 = -1
    best_lr_model = None
    best_vectorizer = None

    for tfcfg in ParameterGrid(tfidf_grid):
        vectorizer = TfidfVectorizer(**tfcfg)
        X_tr = vectorizer.fit_transform(X_train_text)
        X_va = vectorizer.transform(X_val_text)

        for lrcfg in ParameterGrid(lr_grid):
            lr_model = LogisticRegression(random_state=SEED, **lrcfg)
            lr_model.fit(X_tr, y_train)
            val_pred = lr_model.predict(X_va)
            val_f1 = f1_score(y_val, val_pred, average='macro')
            add_tuning_log(prep_name, 'TF-IDF', 'LogisticRegression', {**tfcfg, **lrcfg}, val_f1)
            print(f'LR cfg: {tfcfg} + {lrcfg} -> val_macro_f1={val_f1:.4f}')
            if val_f1 > best_lr_f1:
                best_lr_f1 = val_f1
                best_lr_model = lr_model
                best_vectorizer = vectorizer

    X_te = best_vectorizer.transform(X_test_text)
    test_pred_lr = best_lr_model.predict(X_te)
    evaluate_and_store(y_test, test_pred_lr, prep_name, 'TF-IDF', 'LogisticRegression')

    X_tr_dense = best_vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
    X_va_dense = best_vectorizer.transform(X_val_text).toarray().astype(np.float32)
    X_te_dense = best_vectorizer.transform(X_test_text).toarray().astype(np.float32)

    best_dnn_f1 = -1
    best_dnn_model = None
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    for dcfg in ParameterGrid(dnn_grid):
        dnn = build_tfidf_dnn(
            input_dim=X_tr_dense.shape[1],
            num_classes=num_classes,
            dense1=dcfg['dense1'],
            dense2=dcfg['dense2'],
            drop=dcfg['drop'],
            lr=dcfg['lr']
        )
        dnn.fit(
            X_tr_dense, y_train_oh,
            validation_data=(X_va_dense, y_val_oh),
            epochs=dcfg['epochs'],
            batch_size=dcfg['batch_size'],
            verbose=1,
            callbacks=[early_stop]
        )
        val_pred = np.argmax(dnn.predict(X_va_dense, verbose=0), axis=1)
        val_f1 = f1_score(y_val, val_pred, average='macro')
        add_tuning_log(prep_name, 'TF-IDF', 'DeepNN', dcfg, val_f1)
        print(f'DeepNN cfg: {dcfg} -> val_macro_f1={val_f1:.4f}')
        if val_f1 > best_dnn_f1:
            best_dnn_f1 = val_f1
            best_dnn_model = dnn

    test_pred_dnn = np.argmax(best_dnn_model.predict(X_te_dense, verbose=0), axis=1)
    evaluate_and_store(y_test, test_pred_dnn, prep_name, 'TF-IDF', 'DeepNN')

## 8) Skip-gram + All Required Sequence Models

In [ ]:
MAX_WORDS = 30000
MAX_LEN = 60
EMBED_DIM = 100

def build_embedding_matrix(word_index, w2v, max_words=MAX_WORDS, embed_dim=EMBED_DIM):
    vocab_size = min(max_words, len(word_index) + 1)
    emb_matrix = np.random.normal(scale=0.6, size=(vocab_size, embed_dim))
    emb_matrix[0] = np.zeros(embed_dim)
    for word, idx in word_index.items():
        if idx >= vocab_size:
            continue
        if word in w2v.wv:
            emb_matrix[idx] = w2v.wv[word]
    return emb_matrix

def tokenize_pad(X_train_text, X_val_text, X_test_text):
    tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train_text)

    X_train_seq = tokenizer.texts_to_sequences(X_train_text)
    X_val_seq = tokenizer.texts_to_sequences(X_val_text)
    X_test_seq = tokenizer.texts_to_sequences(X_test_text)

    X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
    X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')
    return tokenizer, X_train_pad, X_val_pad, X_test_pad

def build_seq_model(model_type, vocab_size, emb_matrix, units=64, dropout_rate=0.5, lr=1e-3):
    model = Sequential()
    model.add(Embedding(
        input_dim=vocab_size,
        output_dim=emb_matrix.shape[1],
        weights=[emb_matrix],
        trainable=False
    ))

    if model_type == 'SimpleRNN':
        model.add(SimpleRNN(units))
    elif model_type == 'GRU':
        model.add(GRU(units))
    elif model_type == 'LSTM':
        model.add(LSTM(units))
    elif model_type == 'BiSimpleRNN':
        model.add(Bidirectional(SimpleRNN(units)))
    elif model_type == 'BiGRU':
        model.add(Bidirectional(GRU(units)))
    elif model_type == 'BiLSTM':
        model.add(Bidirectional(LSTM(units)))
    else:
        raise ValueError('Unknown model type')

    model.add(Dropout(dropout_rate))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

seq_model_names = ['SimpleRNN', 'GRU', 'LSTM', 'BiSimpleRNN', 'BiGRU', 'BiLSTM']
seq_grid = {
    'units': [64, 96],
    'dropout_rate': [0.4, 0.5],
    'lr': [1e-3],
    'batch_size': [64],
    'epochs': [6]
}

for prep_name, prep_fn in PREPROCESSORS.items():
    print('\n' + '=' * 95)
    print('Skip-gram experiments for preprocessing:', prep_name)

    X_train_text, X_val_text, X_test_text = get_preprocessed_splits(prep_fn)
    train_sent_tokens = [txt.split() for txt in X_train_text]

    w2v = Word2Vec(
        sentences=train_sent_tokens,
        vector_size=EMBED_DIM,
        window=5,
        min_count=2,
        sg=1,
        workers=4,
        epochs=10,
        seed=SEED
    )

    tokenizer, X_train_pad, X_val_pad, X_test_pad = tokenize_pad(X_train_text, X_val_text, X_test_text)
    vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)
    emb_matrix = build_embedding_matrix(tokenizer.word_index, w2v)

    for model_name in seq_model_names:
        best_f1 = -1
        best_model = None
        early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

        for cfg in ParameterGrid(seq_grid):
            model = build_seq_model(
                model_type=model_name,
                vocab_size=vocab_size,
                emb_matrix=emb_matrix,
                units=cfg['units'],
                dropout_rate=cfg['dropout_rate'],
                lr=cfg['lr']
            )

            model.fit(
                X_train_pad, y_train_oh,
                validation_data=(X_val_pad, y_val_oh),
                epochs=cfg['epochs'],
                batch_size=cfg['batch_size'],
                verbose=1,
                callbacks=[early_stop]
            )

            val_pred = np.argmax(model.predict(X_val_pad, verbose=0), axis=1)
            val_f1 = f1_score(y_val, val_pred, average='macro')
            add_tuning_log(prep_name, 'Skip-gram', model_name, cfg, val_f1)
            print(f'{model_name} cfg: {cfg} -> val_macro_f1={val_f1:.4f}')

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_model = model

        test_pred = np.argmax(best_model.predict(X_test_pad, verbose=0), axis=1)
        evaluate_and_store(y_test, test_pred, prep_name, 'Skip-gram', model_name)

## 9) Tuning Logs + Final Comparison Tables/Plots

In [ ]:
tuning_df = pd.DataFrame(tuning_logs)
results_df = pd.DataFrame(all_results)

print('Tuning log rows:', len(tuning_df))
display(tuning_df.sort_values('val_macro_f1', ascending=False).head(20))

results_df = results_df.sort_values(['macro_f1', 'accuracy'], ascending=False).reset_index(drop=True)
display(results_df)

plt.figure(figsize=(14, 5))
sns.barplot(data=results_df, x='model', y='macro_f1', hue='preprocessing')
plt.title('Macro F1 Comparison by Model and Preprocessing')
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

best_exp = results_df.iloc[0]
worst_exp = results_df.iloc[-1]

print('Best-performing experiment:')
print(best_exp)

print('\nWorst-performing experiment:')
print(worst_exp)

results_df.to_csv('final_results_table.csv', index=False)
tuning_df.to_csv('tuning_logs_table.csv', index=False)
print('\nSaved: final_results_table.csv, tuning_logs_table.csv')

## 10) What to Include in IEEE Report (From this Notebook)

- Abstract: brief approach + strongest finding
- Introduction: task motivation + dataset overview
- Methodology:
  - EDA figures and observations
  - no/extreme/optimum preprocessing rationale
  - TF-IDF and Skip-gram representation details
  - model architecture details and tuning ranges
- Results:
  - per-model Accuracy, macro-F1, confusion matrices, classification reports
  - final comparison plot/table
- Conclusion:
  - best and worst combinations
  - limitations and future work
- References in IEEE style